# Comunicando com o Hugging Face e Gemini

## Obtendo a credencial (gratuita)

Escolha **um** dos dois provedores. Os dois são gratuitos e os dois já estão na lista de
pré-requisitos da disciplina.

**Opção A - Hugging Face** *(recomendada: você vai precisar dessa conta o curso inteiro)*

1. Abra [este link](https://huggingface.co/settings/tokens/new?ownUserPermissions=inference.serverless.write&tokenType=fineGrained)
   — ele já vem com o tipo **Fine-grained** e a permissão **`Make calls to Inference Providers`**
   marcados. É essa permissão que autoriza a chamada ao LLM; sem ela, a resposta é `401`.
2. Dê um nome (ex.: `colab-teia-llm`) → **Create token** → **copie agora**: o valor só aparece
   uma vez.
3. No Colab, clique na **chave 🔑** na barra lateral esquerda (*Secrets*).
4. `+ Adicionar novo secret` → nome exatamente **`HF_TOKEN`** → cole o token.
5. Ligue o botão **Acesso ao notebook**. *(É o passo que todo mundo esquece.)*

**Opção B - Google AI Studio**

1. <https://aistudio.google.com/apikey> → gere uma chave (login com conta Google).
2. Mesmos passos 3 a 5 acima, com o nome **`GOOGLE_API_KEY`**.

> **Nunca** cole a credencial direto numa célula de código. Notebook vai para o GitHub; chave vazada é chave cancelada — o HF varre repositórios públicos e revoga sozinho.

> As duas camadas gratuitas têm limite (o HF dá um crédito mensal; o Google, uma cota por minuto). Nosso laboratório faz **2 chamadas** no total, então cabe folgado em qualquer uma das duas. Se uma cair no meio da aula, troque o valor de `PROVIDER` na célula abaixo e siga.

In [ ]:
PROVIDER = "gemini"      # ou "gemini" — troque aqui se o outro estiver fora do ar

def get_secret(nome):
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except Exception:                    # fora do Colab, digita na hora
        import getpass
        return getpass.getpass(f"Cole seu {nome}: ")


if PROVIDER == "huggingface":
    from huggingface_hub import InferenceClient
    MODEL = "meta-llama/Llama-3.3-70B-Instruct"          # catálogo muda: confira em huggingface.co/models
    cliente = InferenceClient(api_key=get_secret("HF_TOKEN"))

    def chamar_llm(prompt: str) -> str:
        """Chamada mínima a um LLM. Vira perguntar() multi-backend na E04."""
        r = cliente.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,                     # sem isso a resposta corta no meio do JSON
        )
        return r.choices[0].message.content

elif PROVIDER == "gemini":
    from google import genai
    MODEL = "gemini-2.5-flash"                  # catálogo muda: confira em ai.google.dev/models
    cliente = genai.Client(api_key=get_secret("GOOGLE_API_KEY"))

    def chamar_llm(prompt: str) -> str:
        """Chamada mínima a um LLM. Vira perguntar() multi-backend na E04."""
        return cliente.models.generate_content(model=MODEL, contents=prompt).text


try:
    print(f"[{PROVIDER} · {MODEL}]", chamar_llm("Responda em uma frase: o que é NLP?")) # Testando o acesso
except Exception as e:
    print("A chamada falhou:", type(e).__name__)
    print("  429 -> cota ou crédito esgotado; troque o PROVIDER ou gere credencial nova")
    print("  401 / 403 -> credencial inválida, ou o secret está sem 'Acesso ao notebook'")
    raise

In [ ]:
output = chamar_llm("O que é o IFPE?")
print(output)